# Data Sampling: 568K+ Reviews → 300-Review Balanced Sample

This notebook recreates the project sample from the full Amazon Fine Food Reviews dataset.

**Sampling design**
- 10 selected products
- 10 negative reviews per product
- 10 neutral reviews per product
- 10 positive reviews per product
- 30 reviews per product
- 300 reviews total

The sample uses `random_state=42` so the random selection is reproducible.

In [ ]:
import pandas as pd

DATA_FILE = "Reviews.csv"

PRODUCTS = [
    "B007JFMH8M", "B002QWP89S", "B003B3OOPA", "B001EO5Q64", "B001RVFERK",
    "B000KV61FC", "B0013A0QXC", "B005K4Q1YA", "B0090X8IPM", "B007Y59HVM"
]

In [ ]:
def rating_sentiment(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative" 

In [ ]:
df = pd.read_csv(
    DATA_FILE,
    usecols=["ProductId", "Score", "Text"]
)

df = df.dropna()
df = df.drop_duplicates(subset=["ProductId", "Text"])
df = df[df["ProductId"].isin(PRODUCTS)].copy()

df["RatingSentiment"] = df["Score"].apply(rating_sentiment)

print("Rows available after cleaning:", len(df))

In [ ]:
sample_list = []

for product in PRODUCTS:
    product_reviews = df[df["ProductId"] == product]

    for sentiment in ["negative", "neutral", "positive"]:
        group = product_reviews[
            product_reviews["RatingSentiment"] == sentiment
        ]

        sample = group.sample(
            n=10,
            random_state=42
        )

        sample_list.append(sample)

project_sample = pd.concat(
    sample_list,
    ignore_index=True
)

In [ ]:
project_sample = project_sample[
    ["ProductId", "Score", "Text"]
]

project_sample.to_csv(
    "amazon_reviews_project_sample.csv",
    index=False
)

print("Project sample:", len(project_sample), "reviews")

In [ ]:
check = project_sample.copy()
check["RatingSentiment"] = check["Score"].apply(rating_sentiment)

print("\nReviews per product:")
print(check["ProductId"].value_counts())

print("\nSentiment counts:")
print(check["RatingSentiment"].value_counts())